# Causal-intervention experiments

This notebook compares three intervention methods across the Architecture × Objective × Board-size design: **Li et al. activation optimization**, **Nanda et al. linear branch patching**, and the **adapted intervention** used in this study.

It can run either one selected condition or the complete experimental grid. For JEPA, `JEPA_READOUT="best"` selects Linear or MLP exclusively from validation-set legal-move probability mass. Explicit `linear`, `mlp`, and `both` runs remain available. AR always uses its native prediction head.

The adapted method now reports two random controls. The primary geometry-matched control preserves both every layer-wise norm and all cross-layer angles through one random signed permutation per case. The older independent-layer control is retained only as a sensitivity check.


In [1]:
# Execution scope.
RUN_SCOPE = "all"  # @param ["single", "all"]

# Used when RUN_SCOPE == "single".
ARCHITECTURE = "transformer"  # @param ["transformer", "mamba"]
OBJECTIVE = "jepa"  # @param ["ar", "jepa"]
BOARD_SIZE = "8"  # @param [8, 12, 16]
BOARD_SIZE = int(BOARD_SIZE)

RUN_PROFILE = "full"  # @param ["smoke", "full"]
JEPA_READOUT = "best"  # @param ["best", "linear", "mlp", "both"]
METHODS = ("li", "nanda", "adapted")

ALLOW_SOURCE_DRIFT = True  # @param {type:"boolean"}
FORCE = False  # @param {type:"boolean"}
DRY_RUN = False  # @param {type:"boolean"}

ALL_CONDITIONS = [
    (architecture, objective, board_size)
    for board_size in (8, 12, 16)
    for architecture in ("transformer", "mamba")
    for objective in ("ar", "jepa")
]
CONDITIONS = (
    [(ARCHITECTURE.strip().lower(), OBJECTIVE.strip().lower(), BOARD_SIZE)]
    if RUN_SCOPE == "single"
    else ALL_CONDITIONS
)


In [2]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code_Final")
ARTIFACTS_ROOT = Path("/content/drive/MyDrive/Master_Thesis_Artifacts")
assert (PROJECT_ROOT / "src" / "othello_thesis").is_dir(), PROJECT_ROOT
sys.path.insert(0, str(PROJECT_ROOT / "src"))

assert RUN_SCOPE in {"single", "all"}
assert RUN_PROFILE in {"smoke", "full"}
assert JEPA_READOUT in {"best", "linear", "mlp", "both"}
assert METHODS and set(METHODS) <= {"li", "nanda", "adapted"}
assert all(a in {"transformer", "mamba"} for a, _, _ in CONDITIONS)
assert all(o in {"ar", "jepa"} for _, o, _ in CONDITIONS)
assert all(b in {8, 12, 16} for _, _, b in CONDITIONS)

# Install the Mamba dependencies once when at least one selected condition needs them.
if any(architecture == "mamba" for architecture, _, _ in CONDITIONS):
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "packaging", "ninja",
            "causal-conv1d==1.6.2.post1",
            "mamba-ssm==2.3.2.post1",
            "--no-build-isolation",
        ],
        check=True,
    )

from othello_thesis.evaluation.causal_suite import (
    causal_claim_scope,
    resolve_causal_readout_runs,
)

for architecture, objective, board_size in CONDITIONS:
    print(
        f"{architecture}-{objective} {board_size}×{board_size}",
        "| scope:", causal_claim_scope(architecture, objective, board_size),
        "| readout run(s):", resolve_causal_readout_runs(objective, JEPA_READOUT),
    )


Mounted at /content/drive
transformer-ar 8×8 | scope: published_method_replication | readout run(s): ('native',)
transformer-jepa 8×8 | scope: objective_extension | readout run(s): ('best',)
mamba-ar 8×8 | scope: architecture_extension | readout run(s): ('native',)
mamba-jepa 8×8 | scope: architecture_objective_extension | readout run(s): ('best',)
transformer-ar 12×12 | scope: board_extension | readout run(s): ('native',)
transformer-jepa 12×12 | scope: objective_board_extension | readout run(s): ('best',)
mamba-ar 12×12 | scope: architecture_board_extension | readout run(s): ('native',)
mamba-jepa 12×12 | scope: architecture_objective_board_extension | readout run(s): ('best',)
transformer-ar 16×16 | scope: board_extension | readout run(s): ('native',)
transformer-jepa 16×16 | scope: objective_board_extension | readout run(s): ('best',)
mamba-ar 16×16 | scope: architecture_board_extension | readout run(s): ('native',)
mamba-jepa 16×16 | scope: architecture_objective_board_extension |

## Cost and resume contract

Use `smoke` to validate a new configuration and `full` for reported results. The full profile uses 1,000 held-out interventions and Li's 1,000-step optimizer. Intermediate artifacts are saved per method and, for Li, per batch. With `FORCE=False`, rerunning the notebook reuses compatible completed stages and continues incomplete ones.

When a compatible suite-v2 artifact from the earlier random-control protocol is found, the runner archives that adapted result, preserves the completed Li and Nanda stages, and recomputes only the inexpensive adapted stage. Keep `FORCE=False` for this migration.


In [3]:
def run_streaming(command):
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=-1,
    )
    assert process.stdout is not None
    while True:
        chunk = process.stdout.read1(8192)
        if chunk:
            print(chunk.decode("utf-8", errors="replace"), end="", flush=True)
        elif process.poll() is not None:
            break
    if process.wait() != 0:
        raise subprocess.CalledProcessError(process.returncode, command)


completed = []
for index, (architecture, objective, board_size) in enumerate(CONDITIONS, start=1):
    label = f"{architecture}-{objective} {board_size}×{board_size}"
    print("\n" + "=" * 88)
    print(f"[{index}/{len(CONDITIONS)}] {label}")
    print("=" * 88)

    command = [
        sys.executable, "-u",
        str(PROJECT_ROOT / "scripts" / "run_causal_intervention.py"),
        "--architecture", architecture,
        "--objective", objective,
        "--board-size", str(board_size),
        "--profile", RUN_PROFILE,
        "--jepa-readout", JEPA_READOUT,
        "--methods", *METHODS,
        "--artifacts-root", str(ARTIFACTS_ROOT),
        "--project-root", str(PROJECT_ROOT),
    ]
    if ALLOW_SOURCE_DRIFT:
        command.append("--allow-source-drift")
    if FORCE:
        command.append("--force")
    if DRY_RUN:
        command.append("--dry-run")

    run_streaming(command)
    completed.append(label)

print("\nCompleted conditions:")
for label in completed:
    print(" -", label)



[1/12] transformer-ar 8×8
STANDALONE CAUSAL-INTERVENTION SUITE
case       : Transformer-AR 8x8
profile    : full
methods    : li, nanda, adapted
JEPA readout policy: best
readout runs: native
condition scope: published_method_replication
claim scope    : published_method_replication
checkpoint : /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Transformer-AR/transformer_ar_b8_bf16/final.pt
data       : /content/drive/MyDrive/Master_Thesis_Artifacts/data/othello_8x8
output [native]: /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Transformer-AR/transformer_ar_b8_bf16/causal_intervention/causal_intervention_suite_v2/full/readout_native
FULL uses Li's 1,000 activation-optimization steps and is resumable per case batch. Li runs last and prints a timing projection.

READOUT RUN: NATIVE

[NANDA] COMPLETE — reusing saved result

[ADAPTED] START
intervention selection alpha=1 target_mass=0.9283 topN_error=1.000
intervention selection alpha=2 target_mass=0.9716 topN_error=0.300
interven

## Results

The following cell reads the saved JSON artifacts without loading a model or repeating an intervention. Deltas compare each targeted intervention with the unedited (`null`) forward pass on the same counterfactual cases.


In [4]:
import json
from IPython.display import Markdown, display

from othello_thesis.evaluation.causal_suite import (
    ADAPTED_PROTOCOL_ID,
    ADAPTED_RANDOM_CONTROL_ID,
    CAUSAL_SUITE_ID,
    causal_output_subdir,
    resolve_causal_readout_runs,
)
from othello_thesis.evaluation.thesis import resolve_case, resolve_run_dir


def percent(value):
    return f"{100.0 * float(value):.2f}%"


def percentage_point_delta(value):
    return f"{100.0 * float(value):+.2f} pp"


if DRY_RUN:
    print("DRY_RUN produced no results to display.")
else:
    method_variants = {
        "li": "optimized_activation",
        "nanda": "target_class_direction",
        "adapted": "probe_direction",
    }
    method_labels = {"li": "Li", "nanda": "Nanda", "adapted": "Adapted"}

    for architecture, objective, board_size in CONDITIONS:
        selected_case = resolve_case(architecture, objective, board_size)
        run_dir = resolve_run_dir(selected_case, ARTIFACTS_ROOT)
        for readout_run in resolve_causal_readout_runs(objective, JEPA_READOUT):
            results_dir = run_dir / causal_output_subdir(RUN_PROFILE, readout_run)
            results_path = results_dir / f"results__{CAUSAL_SUITE_ID}.json"
            summary_path = results_dir / f"summary__{CAUSAL_SUITE_ID}.md"
            if not results_path.is_file():
                raise FileNotFoundError(results_path)

            state = json.loads(results_path.read_text(encoding="utf-8"))
            identity = state["identity"]
            readout = identity.get("readout_selection", {})
            adapted_result = state.get("methods", {}).get("adapted")
            if adapted_result is not None:
                if adapted_result.get("protocol") != ADAPTED_PROTOCOL_ID:
                    raise RuntimeError(
                        "The adapted result uses the superseded random control. "
                        "Run the execution cell with FORCE=False to migrate it."
                    )
                if identity.get("adapted_random_control") != ADAPTED_RANDOM_CONTROL_ID:
                    raise RuntimeError("Causal identity does not record the corrected random control")

            readout_text = (
                f"**Readout:** `{readout.get('selected', 'native')}`; "
                f"policy `{readout.get('requested_policy', 'native')}`.  "
            )
            lines = [
                f"### {identity['architecture'].title()}-{identity['objective'].upper()} "
                f"{identity['board_size']}×{identity['board_size']} — {identity['profile']}",
                "",
                readout_text,
                "| Method | Cases | Target top-1 legal | Δ top-1 | Target legal mass | Δ target mass | Top-N error | Δ error |",
                "|---|---:|---:|---:|---:|---:|---:|---:|",
            ]
            for method in ("li", "nanda", "adapted"):
                result = state["methods"].get(method)
                if result is None:
                    lines.append(f"| {method_labels[method]} | — | **INCOMPLETE** | — | — | — | — | — |")
                    continue
                baseline = result["test"]["null"]
                edited = result["test"][method_variants[method]]
                lines.append(
                    f"| {method_labels[method]} | {edited['n_cases']} | "
                    f"{percent(edited['target_top1_legal'])} | "
                    f"{percentage_point_delta(edited['target_top1_legal'] - baseline['target_top1_legal'])} | "
                    f"{percent(edited['target_legal_probability_mass'])} | "
                    f"{percentage_point_delta(edited['target_legal_probability_mass'] - baseline['target_legal_probability_mass'])} | "
                    f"{edited['mean_topn_false_positive_plus_false_negative']:.3f} | "
                    f"{edited['mean_topn_false_positive_plus_false_negative'] - baseline['mean_topn_false_positive_plus_false_negative']:+.3f} |"
                )

            if adapted_result is not None:
                targeted = adapted_result["test"]["probe_direction"]
                geometry = adapted_result["test"]["geometry_matched_random"]
                independent = adapted_result["test"].get("independent_layer_random")
                lines += [
                    "",
                    "**Adapted controls.** The primary comparison uses the geometry-matched "
                    "random control; the independent-layer control is shown only for sensitivity.",
                    "",
                    "| Control comparison | Δ top-1 | Δ target mass | Δ Top-N error |",
                    "|---|---:|---:|---:|",
                    f"| Targeted − geometry-matched random | "
                    f"{percentage_point_delta(targeted['target_top1_legal'] - geometry['target_top1_legal'])} | "
                    f"{percentage_point_delta(targeted['target_legal_probability_mass'] - geometry['target_legal_probability_mass'])} | "
                    f"{targeted['mean_topn_false_positive_plus_false_negative'] - geometry['mean_topn_false_positive_plus_false_negative']:+.3f} |",
                ]
                if independent is not None:
                    lines.append(
                        f"| Targeted − independent-layer random | "
                        f"{percentage_point_delta(targeted['target_top1_legal'] - independent['target_top1_legal'])} | "
                        f"{percentage_point_delta(targeted['target_legal_probability_mass'] - independent['target_legal_probability_mass'])} | "
                        f"{targeted['mean_topn_false_positive_plus_false_negative'] - independent['mean_topn_false_positive_plus_false_negative']:+.3f} |"
                    )

            display(Markdown("\n".join(lines)))
            print(f"Saved JSON   : {results_path}")
            print(f"Saved summary: {summary_path}")


### Transformer-AR 8×8 — full

**Readout:** `native_ar`; policy `not_applicable`.  
| Method | Cases | Target top-1 legal | Δ top-1 | Target legal mass | Δ target mass | Top-N error | Δ error |
|---|---:|---:|---:|---:|---:|---:|---:|
| Li | 1000 | 98.50% | +5.40 pp | 93.04% | +2.29 pp | 1.268 | -0.934 |
| Nanda | 1000 | 94.30% | +1.20 pp | 91.14% | +0.39 pp | 1.694 | -0.508 |
| Adapted | 1000 | 99.80% | +6.70 pp | 97.12% | +6.37 pp | 0.340 | -1.862 |

**Adapted controls.** The primary comparison uses the geometry-matched random control; the independent-layer control is shown only for sensitivity.

| Control comparison | Δ top-1 | Δ target mass | Δ Top-N error |
|---|---:|---:|---:|
| Targeted − geometry-matched random | +7.20 pp | +6.51 pp | -1.900 |
| Targeted − independent-layer random | +7.60 pp | +6.38 pp | -1.886 |

Saved JSON   : /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Transformer-AR/transformer_ar_b8_bf16/causal_intervention/causal_intervention_suite_v2/full/readout_native/results__causal_intervention_suite_v2.json
Saved summary: /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Transformer-AR/transformer_ar_b8_bf16/causal_intervention/causal_intervention_suite_v2/full/readout_native/summary__causal_intervention_suite_v2.md


### Transformer-JEPA 8×8 — full

**Readout:** `linear`; policy `best`.  
| Method | Cases | Target top-1 legal | Δ top-1 | Target legal mass | Δ target mass | Top-N error | Δ error |
|---|---:|---:|---:|---:|---:|---:|---:|
| Li | 1000 | 97.20% | +4.80 pp | 92.45% | +2.30 pp | 1.792 | -0.688 |
| Nanda | 1000 | 93.80% | +1.40 pp | 90.72% | +0.57 pp | 2.156 | -0.324 |
| Adapted | 1000 | 98.90% | +6.50 pp | 96.11% | +5.97 pp | 0.364 | -2.116 |

**Adapted controls.** The primary comparison uses the geometry-matched random control; the independent-layer control is shown only for sensitivity.

| Control comparison | Δ top-1 | Δ target mass | Δ Top-N error |
|---|---:|---:|---:|
| Targeted − geometry-matched random | +7.50 pp | +7.28 pp | -2.106 |
| Targeted − independent-layer random | +7.70 pp | +6.14 pp | -2.124 |

Saved JSON   : /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Transformer-JEPA/jepa_v5_infonce_b8_run_001_hd_allpos_bf16/causal_intervention/causal_intervention_suite_v2/full/readout_best/results__causal_intervention_suite_v2.json
Saved summary: /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Transformer-JEPA/jepa_v5_infonce_b8_run_001_hd_allpos_bf16/causal_intervention/causal_intervention_suite_v2/full/readout_best/summary__causal_intervention_suite_v2.md


### Mamba-AR 8×8 — full

**Readout:** `native_ar`; policy `not_applicable`.  
| Method | Cases | Target top-1 legal | Δ top-1 | Target legal mass | Δ target mass | Top-N error | Δ error |
|---|---:|---:|---:|---:|---:|---:|---:|
| Li | 1000 | 93.50% | +0.70 pp | 91.20% | +0.01 pp | 2.124 | -0.140 |
| Nanda | 1000 | 93.00% | +0.20 pp | 91.20% | +0.01 pp | 2.240 | -0.024 |
| Adapted | 1000 | 99.20% | +6.40 pp | 97.54% | +6.35 pp | 0.230 | -2.034 |

**Adapted controls.** The primary comparison uses the geometry-matched random control; the independent-layer control is shown only for sensitivity.

| Control comparison | Δ top-1 | Δ target mass | Δ Top-N error |
|---|---:|---:|---:|
| Targeted − geometry-matched random | +8.60 pp | +7.12 pp | -2.076 |
| Targeted − independent-layer random | +7.10 pp | +6.44 pp | -2.040 |

...

Saved JSON   : /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Transformer-JEPA/transformer_jepa_v5_hd_infonce_allpos_b16/causal_intervention/causal_intervention_suite_v2/full/readout_best/results__causal_intervention_suite_v2.json
Saved summary: /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Transformer-JEPA/transformer_jepa_v5_hd_infonce_allpos_b16/causal_intervention/causal_intervention_suite_v2/full/readout_best/summary__causal_intervention_suite_v2.md


### Mamba-AR 16×16 — full

**Readout:** `native_ar`; policy `not_applicable`.  
| Method | Cases | Target top-1 legal | Δ top-1 | Target legal mass | Δ target mass | Top-N error | Δ error |
|---|---:|---:|---:|---:|---:|---:|---:|
| Li | 1000 | 97.00% | +0.40 pp | 89.45% | +0.57 pp | 4.848 | +0.164 |
| Nanda | 1000 | 97.00% | +0.40 pp | 88.89% | +0.01 pp | 4.676 | -0.008 |
| Adapted | 1000 | 99.20% | +2.60 pp | 90.36% | +1.48 pp | 3.870 | -0.814 |

**Adapted controls.** The primary comparison uses the geometry-matched random control; the independent-layer control is shown only for sensitivity.

| Control comparison | Δ top-1 | Δ target mass | Δ Top-N error |
|---|---:|---:|---:|
| Targeted − geometry-matched random | +5.10 pp | +1.73 pp | -1.258 |
| Targeted − independent-layer random | +3.00 pp | +1.47 pp | -0.944 |

Saved JSON   : /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Mamba-AR/mamba_ar_b16/causal_intervention/causal_intervention_suite_v2/full/readout_native/results__causal_intervention_suite_v2.json
Saved summary: /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Mamba-AR/mamba_ar_b16/causal_intervention/causal_intervention_suite_v2/full/readout_native/summary__causal_intervention_suite_v2.md


### Mamba-JEPA 16×16 — full

**Readout:** `linear`; policy `best`.  
| Method | Cases | Target top-1 legal | Δ top-1 | Target legal mass | Δ target mass | Top-N error | Δ error |
|---|---:|---:|---:|---:|---:|---:|---:|
| Li | 1000 | 96.00% | -0.30 pp | 88.50% | -0.08 pp | 4.816 | +0.154 |
| Nanda | 1000 | 96.40% | +0.10 pp | 88.59% | +0.01 pp | 4.624 | -0.038 |
| Adapted | 1000 | 97.60% | +1.30 pp | 89.39% | +0.81 pp | 3.858 | -0.804 |

**Adapted controls.** The primary comparison uses the geometry-matched random control; the independent-layer control is shown only for sensitivity.

| Control comparison | Δ top-1 | Δ target mass | Δ Top-N error |
|---|---:|---:|---:|
| Targeted − geometry-matched random | +2.10 pp | +1.95 pp | -1.472 |
| Targeted − independent-layer random | +0.00 pp | +0.99 pp | -0.978 |

Saved JSON   : /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Mamba-JEPA/mamba_jepa_v5_hd_infonce_allpos_b16/causal_intervention/causal_intervention_suite_v2/full/readout_best/results__causal_intervention_suite_v2.json
Saved summary: /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Mamba-JEPA/mamba_jepa_v5_hd_infonce_allpos_b16/causal_intervention/causal_intervention_suite_v2/full/readout_best/summary__causal_intervention_suite_v2.md
